# ST-OMR Meter V4-0 Numerator Representation Audit

Zero-training diagnostic: 27 positive Teacher Gold TRAIN families only, numerator-only crop, family-disjoint 3-fold normalized-centroid OOF probe.

No D10 access. No PyTorch runtime requirement. No optimizer. Teacher Gold adaptation-validation is not evaluated. TEST remains sealed.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
from pathlib import Path
import json, shutil, subprocess, sys

REPO_URL = "https://github.com/khfy7wpr5p-maker/st-omr-training.git"
REPO_REF = "fix/meter-v4-0-numerator-representation-audit"
WORK_ROOT = Path("/content/st-omr-meter-v4-0-audit")
REPO_DIR = WORK_ROOT / "repo"

PILOT_ROOT = Path(
    "/content/drive/MyDrive/TEST/METER_V1/00_AUDIT/"
    "teacher_gold_pilot_v1"
)

if WORK_ROOT.exists():
    shutil.rmtree(WORK_ROOT)
WORK_ROOT.mkdir(parents=True)

subprocess.run(
    ["git", "clone", "--branch", REPO_REF, "--single-branch", "--filter=blob:none",
     REPO_URL, str(REPO_DIR)],
    check=True,
)
repository_sha = subprocess.run(
    ["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"],
    check=True, capture_output=True, text=True,
).stdout.strip()

if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

OUTPUT_ROOT = Path(
    "/content/drive/MyDrive/TEST/METER_V1/02_ADAPTATION_RUNS"
) / f"meter-v4-0-numerator-representation-audit-{repository_sha[:12]}"

required = [
    PILOT_ROOT / "pilot-data.json",
    PILOT_ROOT / "ST_OMR_METER_TEACHER_GOLD_PILOT_choices.json",
    PILOT_ROOT / "meter-training-permission-evidence-v1.json",
    PILOT_ROOT / "meter-privacy-review-evidence-v1.json",
]
missing = [str(path) for path in required if not path.is_file()]
if missing:
    raise FileNotFoundError("Missing approved Teacher Gold input(s): " + repr(missing))

print(json.dumps({
    "experiment": "meter-v4-0-numerator-representation-audit-v1",
    "repository_sha": repository_sha,
    "output_root": str(OUTPUT_ROOT),
    "d10_opened": False,
    "optimizer_steps": 0,
    "teacher_adaptation_validation_evaluated": False,
    "test_opened": False,
}, indent=2))


In [ ]:
from st_omr_training.meter_v4_0_numerator_audit_run import run_meter_v4_0_numerator_audit

if OUTPUT_ROOT.exists():
    result_path = OUTPUT_ROOT / "result.json"
    complete_path = OUTPUT_ROOT / "COMPLETE"
    if not result_path.is_file() or not complete_path.is_file():
        raise RuntimeError("Existing V4-0 output is incomplete; move it aside explicitly before rerun.")
    result = json.loads(result_path.read_text(encoding="ascii"))
    if result.get("repository_sha") != repository_sha:
        raise RuntimeError("Existing V4-0 result belongs to a different repository SHA.")
    reused_existing = True
else:
    result = run_meter_v4_0_numerator_audit(
        pilot_path=PILOT_ROOT / "pilot-data.json",
        choices_path=PILOT_ROOT / "ST_OMR_METER_TEACHER_GOLD_PILOT_choices.json",
        permission_path=PILOT_ROOT / "meter-training-permission-evidence-v1.json",
        privacy_path=PILOT_ROOT / "meter-privacy-review-evidence-v1.json",
        output_root=OUTPUT_ROOT,
        repository_sha=repository_sha,
    )
    reused_existing = False

print("REUSED EXISTING:", reused_existing)
print("RESULT:", OUTPUT_ROOT / "result.json")


In [ ]:
print("==============================================")
print("V4-0 OOF SUMMARY")
print("==============================================")
print(json.dumps(result["oof_summary"], indent=2, ensure_ascii=False))

print("\n==============================================")
print("V4-0 DECISION")
print("==============================================")
print(json.dumps(result["decision"], indent=2, ensure_ascii=False))

errors = [row for row in result["oof_predictions"] if not row["correct"]]
print("\n==============================================")
print("OOF ERRORS")
print("==============================================")
print(json.dumps(errors, indent=2, ensure_ascii=False))

print("\n==============================================")
print("SAFETY")
print("==============================================")
print(json.dumps({
    "optimizer_steps": result["optimizer_steps"],
    "d10_opened": result["audit_surface"]["d10_opened"],
    "teacher_adaptation_validation_evaluated": result["audit_surface"]["teacher_adaptation_validation_evaluated"],
    "teacher_adaptation_validation_images_decoded": result["audit_surface"]["teacher_adaptation_validation_images_decoded"],
    "test_opened": result["audit_surface"]["test_opened"],
    "runtime_connected": result["runtime_connected"],
    "production_promotion_authorized": result["production_promotion_authorized"],
}, indent=2))


In [ ]:
from IPython.display import Image as IPyImage, display
sheet = OUTPUT_ROOT / "numerator-crops-contact-sheet.png"
print("CONTACT SHEET:", sheet)
display(IPyImage(filename=str(sheet)))
